# Stage 8: FastAPI integration

These examples use FastAPI TestClient and the existing deterministic pipeline with test retrieval data.

In [ ]:
import copy
from tests.api_fakes import (
    GroundedModelClient, SCHOOL_REQUEST, TimeoutModelClient, VALID_SENSOR,
    generator_builder, make_client,
)
client = make_client(builder=generator_builder(GroundedModelClient()))
client.__enter__()
print(client.get('/api/v1/health').json()['status'])

In [ ]:
school_response = client.post('/api/v1/plans', json=SCHOOL_REQUEST)
school_response.raise_for_status()
school_plan = school_response.json()['plan']
print(school_plan['plan_id'], school_plan['total_planned_minutes'])

In [ ]:
sensor_response = client.post('/api/v1/plans/adapt', json={
    'plan': school_plan, 'sensor_observation': VALID_SENSOR,
})
sensor_response.raise_for_status()
adapted_plan = sensor_response.json()['adapted_plan']
print(adapted_plan['adaptation_applied'], adapted_plan['severity'])

In [ ]:
full_request = copy.deepcopy(SCHOOL_REQUEST)
full_request['sensor_observation'] = VALID_SENSOR
full_response = client.post('/api/v1/plans/full', json=full_request)
full_response.raise_for_status()
print(full_response.json()['adapted_plan']['adapted_plan_id'])

In [ ]:
explanation = client.post('/api/v1/explanations', json={
    'plan': school_plan, 'adapted_plan': adapted_plan,
})
explanation.raise_for_status()
print(explanation.json()['generation_mode'])

In [ ]:
fallback_client = make_client(builder=generator_builder(TimeoutModelClient()))
fallback_client.__enter__()
fallback_plan = fallback_client.post('/api/v1/plans', json=SCHOOL_REQUEST).json()['plan']
fallback = fallback_client.post('/api/v1/explanations', json={'plan': fallback_plan})
print(fallback.status_code, fallback.json()['fallback_reason_code'])

In [ ]:
validation_error = client.post('/api/v1/plans', json={
    'total_available_minutes': 10, 'subjects': [],
})
print(validation_error.status_code, validation_error.json()['error']['code'])
fallback_client.__exit__(None, None, None)
client.__exit__(None, None, None)